# 02 — Árvore determinística de bola morta

Cada frame passa por uma **cascata de nós legíveis**, e o primeiro nó que dispara decide se a bola está morta. No fim, uma suavização temporal tira o
"pisca-pisca" e o resultado é avaliado com as métricas oficiais (`evaluation/possession_metrics.py`).

**Regras do jogo, para não nos enganarmos:**

1. **Os limiares são ajustados só no rótulo fraco** (Smart Tagging, jogo inteiro, sem a janela do GT ± 1 min). O GT público serve só para *avaliar*.
2. Com 4 trechos D no GT, qualquer diferença de 1–2 trechos já move as métricas. **Diferenças pequenas são ruído**, como o próprio `EVALUATION.md` avisa.
3. A seção 7 varre os limiares *olhando o GT*. Ela está marcada como **diagnóstico otimista** e não serve para escolher nada.

Pré-requisitos: `python viewer/features.py` e `python analise/features_coletivas.py`. O notebook 01 é recomendado, mas não obrigatório.

In [ ]:
%matplotlib inline
import os, sys, json, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

HERE = os.path.abspath("") if os.path.exists("arvore.py") else os.path.join(os.path.abspath(""), "analise")
sys.path.insert(0, HERE)
import rotulos as R
import arvore as T
from features_coletivas import load_all

X, meta = load_all()
I, N, FPS = meta["index"], meta["n_frames"], R.FPS
F, I2 = T.stack(X, I, T.context_features(X, I))      # + janelas centradas e futuras
f = lambda name: F[:, I2[name]]

gt = R.gt_public(); cov = gt != "X"; yD = gt == "D"; graded = R.graded_mask(N)
segs = R.gt_dead_segments(gt)
gt_window = np.zeros(N, bool); gt_window[np.flatnonzero(cov)[0] - 60 * FPS:np.flatnonzero(cov)[-1] + 60 * FPS] = True
base = T.load_baseline("baseline_smooth50")
gt_team = np.where(np.isin(gt, ["H", "A"]), gt, base)   # times do GT onde houver (diagnóstico)

OUT = os.path.join(HERE, "out", "nb02"); os.makedirs(OUT, exist_ok=True)
sns.set_theme(style="whitegrid")
COL = {"H": "#a50044", "A": "#1f77b4", "D": "#111111"}
minute = lambda fr: np.asarray(fr) / FPS / 60
print("{} features (+{} de contexto), GT: {} frames, {} trechos D".format(X.shape[1], F.shape[1] - X.shape[1], cov.sum(), len(segs)))

## 1. Onde a bola morta pesa na métrica

Antes de construir qualquer coisa, vale medir quanto um detector de D *pode* ajudar. Comparo três referências, cada uma combinada com
os H/A do `baseline_smooth50`:

| referência | o que é |
|---|---|
| `baseline` | o próprio baseline, que diz D quando a bola do ATD some |
| `nunca D` | nunca diz D: todo frame recebe o último time |
| `oráculo D` | o D exatamente igual ao do GT, o limite superior |

Também combino cada variante com os **times do GT** no lugar dos do baseline. Isso isola a qualidade do detector de D: o erro de H/A
deixa de contar, e o que sobra é só o efeito do D.

In [ ]:
def prf(dead):
    tp = (dead & yD)[cov].sum(); fp = (dead & ~yD)[cov].sum(); fn = (~dead & yD)[cov].sum()
    p = tp / max(1, tp + fp); r = tp / max(1, tp + fn)
    return p, r, 2 * p * r / max(1e-9, p + r)

ROWS = []
def avaliar(nome, dead, guardar=True):
    a = T.evaluate(gt, T.combine_with_baseline(base, dead, graded))
    b = T.evaluate(gt, T.combine_with_baseline(gt_team, dead, graded))
    p, r, f1 = prf(dead)
    row = dict(variante=nome, precisao_D=p, recall_D=r, F1_D=f1,
               spellF1=a["spell_f1_mean"], macroF1=a["macro_f1"], alive_dead_F1=a["alive_dead_f1"],
               spellF1_timeGT=b["spell_f1_mean"], macroF1_timeGT=b["macro_f1"],
               posses_prev=b["n_pred_spells"], posses_gt=b["n_gt_spells"], trechos_D_jogo=len(T.runs(dead & graded)))
    if guardar:
        ROWS[:] = [x for x in ROWS if x["variante"] != nome] + [row]
    return row

def placar(filtro=None):
    d = pd.DataFrame(ROWS)
    if filtro: d = d[d.variante.str.contains(filtro, regex=False) | d.variante.isin(["baseline (bola some)", "nunca D", "oráculo D"])]
    return d.style.format({c: "{:.3f}" for c in d.columns if d[c].dtype.kind == "f"}).background_gradient(
        subset=["spellF1", "macroF1", "spellF1_timeGT"], cmap="Greens")

avaliar("baseline (bola some)", base == "D")
avaliar("nunca D", np.zeros(N, bool))
avaliar("oráculo D", yD.copy())
placar()

**Leitura.**

* Com os times do baseline, o D perfeito leva o spellF1 de 0,33 a 0,40 e o macroF1 de 0,62 a 0,91. O macroF1 é onde o D pesa
  (é uma das três classes). O spellF1 é dominado pela qualidade do H/A.
* Com os times do GT, **"nunca D" já dá spellF1 ≈ 0,89**, mais que o D do baseline (0,79). Um D *falso* no meio de uma posse a parte
  em duas, e as duas metades não casam com a posse real no IoU. **Precisão importa mais do que recall** para o spellF1: um detector
  de D só vale a pena se errar pouco quando diz D.

## 2. Contexto temporal: por que olhar o futuro

O notebook 01 mostrou que a bola morta é **assimétrica no tempo**. O apito é gradual (a velocidade cai ao longo de ~10 s) e o reinício é
abrupto (mínimo de velocidade e depois aceleração). A comparação abaixo, alinhada ao início e ao fim de cada D, mostra a mesma
velocidade vista por janelas de passado, centrada e de futuro:

* **passado** (`speed_past5s_mean`): só fica baixa uns 5 s *depois* do apito, então atrasa o início, e continua baixa depois do reinício, então atrasa o fim;
* **futuro** (`speed_fut5s_mean`): cai *antes* da janela de passado e sobe exatamente no reinício, porque o futuro já inclui a corrida.

Mesmo assim, **no apito nenhuma janela resolve tudo**: os jogadores continuam correndo por alguns segundos depois de a bola sair
(acompanham o lance, reclamam, voltam), então o início do D sai com uns 5 s de atraso. O fim (o reinício) é onde o movimento coletivo é
um relógio preciso.

A submissão é offline, então usar o futuro é legítimo. `ball_gap_len_s`, por exemplo, já olha o futuro.

In [ ]:
names = [("speed_past5s_mean", "passado 5 s"), ("speed_c5s_mean", "centrada 5 s"), ("speed_fut5s_mean", "futuro 5 s")]
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
sp = R.set_pieces()
for ax, (anchors, title) in zip(axes, [([s for s, _ in segs], "início de D (GT)"), ([e for _, e in segs], "reinício (fim de D no GT)")]):
    for (n, lab), c in zip(names, ["#999999", "#4c72b0", "#d62728"]):
        t, M = R.event_aligned(f(n), anchors, 15, 15)
        ax.plot(t, np.nanmean(M, 0), color=c, lw=2, label=lab)
    ax.axvline(0, color="k", ls="--"); ax.axhline(1.2, color="grey", ls=":", lw=0.8)
    ax.set_title(title); ax.set_xlabel("s em relação à âncora"); ax.legend()
axes[0].set_ylabel("velocidade média [m/s]")
fig.suptitle("mesma velocidade, três janelas — média dos 4 eventos do GT"); fig.tight_layout(); plt.show()

ctx = [n for n in I2 if n not in I]
tab = pd.DataFrame({"feature": ctx, "AUC_GT_sep": [abs(R.auc(yD[cov].astype(int), f(n)[cov]) - 0.5) + 0.5 for n in ctx]})
display(tab.sort_values("AUC_GT_sep", ascending=False).head(12).round(3).T)

## 3. A árvore manual: uma cascata determinística

Cada nó é uma regra que um analista consegue ler e que vem de uma hipótese testada no notebook 01:

| nó | regra | hipótese |
|---|---|---|
| **N1** futuro lento | `speed_fut5s_mean < t` | H1: desaceleração coletiva, olhando para a frente |
| **N2** lento e desorganizado | `speed_c5s_mean < t` **e** `pol_c5s_mean < t` | H1 + H2: sem velocidade **e** sem ordem (Vicsek) |
| **N3** bola sumiu na linha e ninguém persegue | `ball_lost_at_line_s > 1` **e** `closing_c5s_mean < t` | H6 + H3: quimiotaxia |
| **N4** times afastados e andando | `teams_centroid_d_m > t` **e** `a_frac_walk > t` | H5: formação de reinício |
| **viva** | nenhum nó disparou | |

**Ajuste.** Os limiares `t` saem do rótulo fraco, um nó por vez e só sobre os frames que *chegaram* até aquele nó. O critério é o maior recall de D
com precisão ≥ 0,9 no treino, com a classe "morta" pesando 3× para compensar a proporção artificial do rótulo fraco.

**Poda.** Um nó cuja precisão no treino fica abaixo de 0,7 é desligado. Um nó assim só criaria D falso, e pela seção 1 D falso é o que mais custa.

**Qual rótulo fraco?** Três versões da regra de "viva" (veja `rotulos.weak_labels`) mudam o resultado. A tabela da seção 3.1 mostra as três.
A escolhida é `jogadores`, que tem a menor contaminação. A ressalva: ela define viva como velocidade (3 s) > 1,2 m/s, então os limiares de velocidade
tendem a cair logo abaixo de 1,2. **É o GT que diz se isso generaliza.**

In [ ]:
NODES = [
    {"id": "N1", "nome": "futuro lento", "regra": [("speed_fut5s_mean", "<", None)]},
    {"id": "N2", "nome": "lento e desorganizado", "regra": [("speed_c5s_mean", "<", None), ("pol_c5s_mean", "<", None)]},
    {"id": "N3", "nome": "bola sumiu na linha, ninguém persegue", "regra": [("ball_lost_at_line_s", ">", 1.0), ("closing_c5s_mean", "<", None)]},
    {"id": "N4", "nome": "times afastados e andando", "regra": [("teams_centroid_d_m", ">", None), ("a_frac_walk", ">", None)]},
]
ALIVE_RULE, MIN_PREC, WEIGHT = "jogadores", 0.9, 3.0
yw, _ = R.weak_labels(X, I, use_line=False, alive_rule=ALIVE_RULE)
train = (yw >= 0) & graded & ~gt_window
print("treino: {} frames viva, {} morta (fora da janela do GT)".format(*np.bincount(yw[train])))

tree = T.RuleTree(NODES, min_precision=MIN_PREC, weight=WEIGHT).fit(F, I2, yw.astype(int), train)
dead_raw, node = tree.predict(F, I2)
desc = tree.describe()
# precisão de cada nó no GT (só avaliação)
desc["frames_GT"] = [int((cov & (node == n)).sum()) for n in desc.no]
desc["precisao_GT"] = [yD[cov & (node == n)].mean() if (cov & (node == n)).any() else np.nan for n in desc.no]
display(desc.round(3))
print("N_viva no GT: {} frames, {:.1%} deles são D (o que a árvore perdeu)".format((cov & (node == "N_viva")).sum(), yD[cov & (node == "N_viva")].mean()))
avaliar("A1 manual (cru)", dead_raw)
dead_manual = T.clean_runs(dead_raw, min_dead_s=3, min_alive_s=2)
avaliar("A2 manual + limpeza", dead_manual)
placar("manual")

### 3.1 Sensibilidade à escolha do rótulo fraco

A mesma cascata, ajustada com as três regras de "viva" e duas precisões alvo. Esta tabela é o principal teste de robustez: se o resultado
só aparece numa configuração, ele é frágil.

In [ ]:
sens = []
for rule in ("jogadores", "bola", "nenhum"):
    y_, _ = R.weak_labels(X, I, use_line=False, alive_rule=rule)
    tr_ = (y_ >= 0) & graded & ~gt_window
    for mp in (0.9, 0.8):
        t_ = T.RuleTree(NODES, min_precision=mp, weight=WEIGHT).fit(F, I2, y_.astype(int), tr_)
        d_, _n = t_.predict(F, I2)
        r = avaliar("sens {} p≥{}".format(rule, mp), T.clean_runs(d_, 3, 2), guardar=False)
        r["nos_ativos"] = ", ".join(n["id"] for n in t_.nodes if not n.get("podado"))
        r["limiar_N1"] = round(t_.nodes[0]["regra"][0][2], 2)
        sens.append(r)
display(pd.DataFrame(sens)[["variante", "nos_ativos", "limiar_N1", "precisao_D", "recall_D", "spellF1", "macroF1", "spellF1_timeGT", "trechos_D_jogo"]].round(3))

**Leitura.**

* Com `nenhum` (viva = só longe de reinício, com 13% de D escondido), nenhum nó chega a 0,7 de precisão e **tudo é podado**. O
  detector fica mudo, o que é o comportamento seguro: na dúvida, não marcar D.
* Com `bola`, o limiar do N1 cai para ~1,0 m/s e o detector vira conservador: precisão ~0,9 e recall baixo.
* Com `jogadores`, o limiar fica em ~1,1 m/s e o detector tem o melhor equilíbrio.
* **Os nós N3 e N4 (perseguição e formação) costumam ser podados.** Nesta cascata, eles não acrescentam precisão depois que a velocidade já decidiu.
  Isso bate com a LDA do notebook 01: as hipóteses H3 e H5 são verdadeiras, mas redundantes com a velocidade para decidir frame a frame.

In [ ]:
# linha do tempo: em que nó cada frame do GT parou
w0, w1 = np.flatnonzero(cov)[[0, -1]]
fr = np.arange(w0, w1 + 1)
node_col = {"N1": "#d62728", "N2": "#ff7f0e", "N3": "#9467bd", "N4": "#8c564b", "N_viva": "#e8e8e8"}
fig, axes = plt.subplots(4, 1, figsize=(15, 4.6), sharex=True, gridspec_kw=dict(height_ratios=[1, 1, 1, 2.2]))
for lab in "HAD":
    for s, e in R.runs(gt[fr] == lab): axes[0].axvspan(minute(fr[s]), minute(fr[min(e, len(fr) - 1)]), color=COL[lab], lw=0)
for n, c in node_col.items():
    for s, e in R.runs(node[fr] == n): axes[1].axvspan(minute(fr[s]), minute(fr[min(e, len(fr) - 1)]), color=c, lw=0)
for s, e in R.runs(dead_manual[fr]): axes[2].axvspan(minute(fr[s]), minute(fr[min(e, len(fr) - 1)]), color="k", lw=0)
axes[3].plot(minute(fr), f("speed_fut5s_mean")[fr], color="#d62728", lw=1, label="speed_fut5s_mean")
axes[3].plot(minute(fr), f("speed_c5s_mean")[fr], color="#4c72b0", lw=1, label="speed_c5s_mean")
axes[3].axhline(tree.nodes[0]["regra"][0][2], color="#d62728", ls="--", lw=0.8, label="limiar N1")
for s, e in segs: axes[3].axvspan(minute(s), minute(e), color="k", alpha=0.12)
axes[3].legend(fontsize=7, ncol=3); axes[3].set_ylabel("m/s")
for ax, lab in zip(axes[:3], ["GT", "nó", "D previsto"]): ax.set_yticks([]); ax.set_ylabel(lab, rotation=0, ha="right")
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=c, label=n) for n, c in node_col.items()], fontsize=7, ncol=5, loc="upper right")
axes[-1].set_xlabel("minuto"); fig.tight_layout(); fig.savefig(os.path.join(OUT, "linha_do_tempo_nos.png"), dpi=120); plt.show()

display(T.dead_segment_report(dead_manual, gt))

## 4. Árvore aprendida (CART) para comparar

A mesma pergunta respondida por um `DecisionTreeClassifier` raso, treinado no mesmo rótulo fraco. Há duas restrições de honestidade:

* **só features de jogadores, perseguição e contexto.** A posição da bola fica de fora, porque as janelas de set piece são quase todas laterais e escanteios
  e a árvore aprenderia "bola perto da linha", que é o viés do rótulo e não o fenômeno;
* `class_weight="balanced"` e `min_samples_leaf=300`, cerca de 1 minuto de jogo com a amostragem 1:5, para impedir folhas que decoram um evento.

Depois comparo com uma árvore treinada **no próprio GT**, com validação deixando um trecho D de fora por vez. É a forma de mostrar o que acontece
quando só há 4 eventos.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

ball_pos = [n for n in I2 if n.startswith("ball_") and n not in ("ball_centroid_cos_2s", "ball_nearest3_d_m")]
excl = set(ball_pos) | {"half", "half_time_s", "reproj_error", "ball_wcentroid_d_m", "h_centroid_x_att", "a_centroid_x_att",
                        "all_frac_own_half", "all_players_in_boxes"}
CANDS = [n for n in I2 if not n.startswith(("st_", "trk_")) and n not in excl]
cols = [I2[c] for c in CANDS]
tr_idx = np.flatnonzero(train)[::5]
print("{} features candidatas, {} amostras de treino".format(len(CANDS), len(tr_idx)))

learned = {}
for depth in (2, 3, 4):
    clf = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=300, class_weight="balanced", random_state=0)
    clf.fit(F[tr_idx][:, cols], yw[tr_idx])
    p = clf.predict_proba(F[:, cols])[:, 1]
    learned[depth] = (clf, p)
    used = sorted({CANDS[i] for i in clf.tree_.feature if i >= 0})
    print("profundidade {}: AUC no GT {:.3f} | usa {}".format(depth, R.auc(yD[cov].astype(int), p[cov]), used))
    avaliar("B{} CART prof. {} (p>0,5 suav. 5 s)".format(depth, depth), T.clean_runs(T.smooth_score(p, 5) > 0.5, 3, 2))
placar("CART")

In [ ]:
clf, p_cart = learned[2]
print(export_text(clf, feature_names=CANDS, decimals=2))
fig, ax = plt.subplots(figsize=(12, 5))
plot_tree(clf, feature_names=CANDS, class_names=["viva", "morta"], filled=True, impurity=False, proportion=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("CART profundidade 2, treinada no rótulo fraco (sem GT)"); plt.show()

⚠️ **A circularidade apareceu.** O primeiro corte da CART é `all_speed_mean_3s ≤ 1,20`, que é *exatamente* a definição de viva
do rótulo fraco `jogadores` (velocidade de 3 s > 1,2). A árvore reaprendeu a regra do rótulo. Isso não invalida o resultado, porque o GT
confirma que a regra generaliza (AUC 0,855). Mas mostra que **a CART não descobriu nada que não estivesse no rótulo**. A cascata manual
é mais informativa porque cada nó é uma hipótese explícita, e a poda diz quais sobrevivem.

### 4.1 O que acontece se treinar no GT? (validação deixando um trecho D de fora)

A janela do GT é cortada em 4 blocos, cada um com um trecho D. Cada árvore treina em 3 blocos e prevê o quarto.

In [ ]:
gidx = np.flatnonzero(cov)
mids = [(segs[i][1] + segs[i + 1][0]) // 2 for i in range(len(segs) - 1)]
group = np.searchsorted(mids, gidx)
loso = []
for depth in (2, 3, 4):
    p = np.full(N, np.nan)
    for k in range(len(segs)):
        trn, tst = gidx[group != k], gidx[group == k]
        m = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=100, class_weight="balanced", random_state=0)
        m.fit(F[trn][:, cols], yD[trn].astype(int))
        p[tst] = m.predict_proba(F[tst][:, cols])[:, 1]
    full = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=100, class_weight="balanced", random_state=0).fit(F[gidx][:, cols], yD[gidx].astype(int))
    loso.append(dict(profundidade=depth, AUC_validacao_por_trecho=R.auc(yD[cov].astype(int), p[cov]),
                     AUC_no_proprio_treino=R.auc(yD[cov].astype(int), full.predict_proba(F[cov][:, cols])[:, 1])))
display(pd.DataFrame(loso).round(3))
print("Comparação: a CART treinada no rótulo fraco (sem ver o GT) tem AUC {:.3f} no GT.".format(R.auc(yD[cov].astype(int), learned[2][1][cov])))

**Leitura.** Treinada no GT, a árvore decora: o AUC no próprio treino é ≈ 1, e deixando um trecho de fora cai bastante. Com 4 eventos não dá
para aprender limiares. Daí a escolha de usar o **rótulo fraco do jogo inteiro (59 set pieces) para ajustar e o GT só para avaliar**.

In [ ]:
from sklearn.inspection import permutation_importance
clf3, _ = learned[3]
sel = np.flatnonzero(cov)[::3]
pi = permutation_importance(clf3, F[sel][:, cols], yD[sel].astype(int), scoring="roc_auc", n_repeats=10, random_state=0)
imp = pd.Series(pi.importances_mean, index=CANDS).sort_values(ascending=False)
imp = imp[imp.abs() > 1e-4]
display(imp.head(10).round(3).to_frame("queda de AUC no GT ao embaralhar"))

## 5. Suavização temporal

A decisão por frame pisca. Testo três pós-processamentos sobre a saída da árvore manual (binária) e da CART (probabilidade):

| método | como funciona |
|---|---|
| **limpeza** | preenche buracos de viva < 2 s dentro de D e apaga trechos D < 3 s |
| **histerese** | média móvel do escore; entra em D acima de `t_on` e sai abaixo de `t_off` |
| **HMM 2 estados** | Viterbi com durações médias explícitas (D ≈ 15 s, viva ≈ 45 s); a emissão é a probabilidade da CART |

Os hiperparâmetros são fixos e razoáveis, não otimizados no GT.

In [ ]:
score_manual = dead_raw.astype(float)
for w, ton, toff in ((3, 0.6, 0.3), (5, 0.5, 0.3)):
    avaliar("C manual histerese w{} on{} off{}".format(w, ton, toff),
            T.clean_runs(T.hysteresis(T.smooth_score(score_manual, w), ton, toff), 3, 2))
p2 = learned[2][1]
for prior in (0.2, 0.3):
    avaliar("C CART2 HMM prior {}".format(prior), T.viterbi2(p2, mean_dead_s=15, mean_alive_s=45, prior_dead=prior))
avaliar("C CART2 histerese w5 on0,6 off0,4", T.clean_runs(T.hysteresis(T.smooth_score(p2, 5), 0.6, 0.4), 3, 2))
placar()

**Leitura.** A histerese sobre a cascata manual empata com a limpeza simples: a diferença é de 1 posse em 17, ou seja, ruído. O HMM vai
mal. A emissão vem da probabilidade da CART com `class_weight="balanced"`, que **não é calibrada** (superestima a bola morta), então o
Viterbi "gruda" em D. Um HMM só valeria a pena com probabilidades calibradas (`CalibratedClassifierCV` no rótulo fraco) e
durações estimadas por tipo de reinício.

## 6. Avaliação oficial: o placar completo

A coluna `spellF1` é a métrica do ranking, com H/A do `baseline_smooth50`. A coluna `spellF1_timeGT` isola o efeito do D. Os trechos D no jogo
são para comparar com a realidade: o Smart Tagging marca 59 set pieces, e um jogo tem por volta de 100–130 paradas.

In [ ]:
board = pd.DataFrame(ROWS).sort_values("spellF1", ascending=False)
board.to_csv(os.path.join(OUT, "placar.csv"), index=False)
display(board.style.format({c: "{:.3f}" for c in board.columns if board[c].dtype.kind == "f"})
        .background_gradient(subset=["spellF1", "macroF1", "spellF1_timeGT", "precisao_D"], cmap="Greens"))

In [ ]:
FINAL = "A2 manual + limpeza"
dead_final = dead_manual
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
dur = np.array([(e - s) / FPS for s, e in T.runs(dead_final & graded)])
axes[0].hist(dur, bins=np.arange(0, 120, 4), color="k"); axes[0].set_xlabel("duração do trecho D previsto [s]")
axes[0].set_title("{} trechos D no jogo (mediana {:.0f} s)".format(len(dur), np.median(dur)))
gtd = np.array([(e - s) / FPS for s, e in segs])
for x in gtd: axes[0].axvline(x, color="red", lw=1)
spf = R.set_pieces().start_frame.values
hit = np.array([dead_final[max(0, s - 5 * FPS):s + 5 * FPS].any() for s in spf])
axes[1].bar(["detectados", "perdidos"], [hit.sum(), (~hit).sum()], color=["#2ca02c", "#d62728"])
axes[1].set_title("59 set pieces: D previsto a ±5 s do start_frame")
by = pd.Series(hit, index=R.set_pieces().type).groupby(level=0).mean()
axes[2].barh(by.index, by.values, color="#4c72b0"); axes[2].set_xlim(0, 1); axes[2].set_title("taxa de detecção por tipo de reinício")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "jogo_inteiro.png"), dpi=110); plt.show()

## 7. Diagnóstico otimista: o limiar do N1 visto pelo GT

**Não use esta seção para escolher limiares.** Ela olha o GT, e com 4 eventos isso é overfitting garantido. Ela serve para mostrar
*onde* o limiar ajustado no rótulo fraco cai em relação à curva no GT e quão estreita é a faixa boa.

In [ ]:
grid = np.round(np.arange(0.7, 1.75, 0.05), 2)
curve = []
for t in grid:
    nodes = [dict(n, regra=[list(c) for c in n["regra"]]) for n in tree.nodes]
    nodes[0]["regra"][0][2] = t
    tt = T.RuleTree(nodes); tt.nodes = nodes
    d_, _ = tt.predict(F, I2)
    r = avaliar("diag", T.clean_runs(d_, 3, 2), guardar=False); r["limiar"] = t; curve.append(r)
curve = pd.DataFrame(curve)
fig, ax = plt.subplots(figsize=(9, 4))
for c, col in (("spellF1", "k"), ("macroF1", "#4c72b0"), ("spellF1_timeGT", "#2ca02c"), ("precisao_D", "#d62728"), ("recall_D", "#ff7f0e")):
    ax.plot(curve.limiar, curve[c], "o-", color=col, label=c, ms=3)
ax.axvline(tree.nodes[0]["regra"][0][2], color="grey", ls="--", label="limiar ajustado no fraco")
ax.set_xlabel("limiar N1: speed_fut5s_mean < t [m/s]"); ax.legend(fontsize=8, ncol=2); ax.set_title("sensibilidade no GT (diagnóstico)")
plt.show()

## 8. Exportar

* `analise/out/arvore_D.csv`: `frame, morta, no, escore_cart` para todos os frames. É a entrada para quem for fazer o classificador de posse.
* `analise/out/sub_arvore_smooth50.csv`: uma submissão completa (H/A do `baseline_smooth50` com o nosso D).

Para pontuar no script oficial: `copy analise\out\sub_arvore_smooth50.csv evaluation\results\` e depois `python evaluation/run_evaluation.py`.
A pasta `results/*.csv` está no `.gitignore`.

In [ ]:
out = pd.DataFrame({"frame": np.arange(N), "morta": dead_final.astype(int), "no": np.where(dead_final, node, "N_viva"),
                    "escore_cart": np.round(learned[2][1], 4)})
out.loc[~graded, ["morta"]] = 0
out.to_csv(os.path.join(HERE, "out", "arvore_D.csv"), index=False)
sub = T.combine_with_baseline(base, dead_final, graded)
pd.DataFrame({"frame": np.arange(N), "label": sub}).to_csv(os.path.join(HERE, "out", "sub_arvore_smooth50.csv"), index=False)
import possession_metrics as pm
err = pm.validate_submission(pm.read_label_csv(os.path.join(HERE, "out", "sub_arvore_smooth50.csv")), N)
print("submissão válida" if err is None else "ERRO: " + err)
with open(os.path.join(HERE, "out", "arvore_regras.json"), "w", encoding="utf-8") as h:
    json.dump({"alive_rule": ALIVE_RULE, "min_precision": MIN_PREC, "weight": WEIGHT, "nos": tree.nodes,
               "limpeza": {"min_dead_s": 3, "min_alive_s": 2}}, h, indent=1, ensure_ascii=False, default=float)
print(tree.describe()[["no", "nome", "regra", "podado"]].to_string(index=False))

## 9. Conclusões

In [ ]:
b = pd.DataFrame(ROWS).set_index("variante")
ref, fin = b.loc["baseline (bola some)"], b.loc[FINAL]
print("Árvore final ({}), comparada com o baseline_smooth50 no GT público:".format(FINAL))
for c in ("spellF1", "macroF1", "alive_dead_F1", "precisao_D", "recall_D", "spellF1_timeGT"):
    print("  {:<16} {:.3f} → {:.3f}  ({:+.3f})".format(c, ref[c], fin[c], fin[c] - ref[c]))

1. **A regra que decide é simples:** "o bloco vai estar lento nos próximos 5 s" (velocidade média < 1,1 m/s), com limiar ajustado no
   rótulo fraco sem tocar no GT. No GT, esse nó acerta **99%** dos frames em que dispara. Contra o `baseline_smooth50`, o spellF1 vai de
   0,326 a 0,410 (o mesmo que o D *oráculo* com esses times) e o macroF1 de 0,62 a 0,79.
2. **O reinício é detectado com precisão; o apito, não.** O fim de cada D sai entre −0,4 e +3 s do GT. O início sai com 5–9 s de atraso,
   e o trecho curto de 5,5 s (D1) não é detectado. Os jogadores continuam se movendo depois da saída da bola, então o
   movimento coletivo sozinho não "ouve" o apito. Para o início, o sinal precisa vir da **bola** (saída pela linha, lacuna, mudança brusca
   de direção ou velocidade).
3. **As hipóteses coletivas (polarização, perseguição, formação) são verdadeiras mas redundantes para decidir frame a frame.** O N2 (lento
   e desorganizado) acrescenta cobertura, mas com precisão menor (0,69 no GT). Os nós de perseguição e formação são **podados** porque não
   chegam à precisão mínima. O lugar natural delas é **por tipo de reinício** (escanteio: convergência para a área; tiro de meta: o bloco se
   alonga), e isso pede mais eventos rotulados.
4. **A precisão manda no spellF1.** D falso parte posses. Por isso a cascata é ajustada com precisão ≥ 0,9 e a poda desliga nós fracos.
   Com os times do GT, "nunca D" já dá 0,886, e a árvore leva a 0,895–0,933.
5. **Limitação estatística.** São 4 trechos D no GT e todo número aqui tem incerteza de ±1 trecho. O próximo passo de maior valor é
   **rotular à mão mais 20–30 paradas** no `viewer/` (início e fim do D), o que permitiria validar por tipo de reinício.
6. **Próximos passos técnicos:**
   * um nó de **início** baseado na bola: a bola do `tracking_bola` (o CSV não existe nesta máquina; `features.py` cria as colunas `trk_*` quando ele existe) e a lacuna perto da linha;
   * um nó de **reinício** explícito: o pico de aceleração coletiva depois de um mínimo (`speed_gain_fut3s`) já está calculado;
   * um classificador de posse melhor que o baseline, porque é o H/A que domina o spellF1.